# Measurement-error-aware covariance

This notebook tests the estimators against known $\Sigma_{\mathrm{true}}$. Under the model assumptions, **observed variability = latent biological variability + measurement variability**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import noisecov as nc

sim = nc.simulate_measurement_error(
    n_cells=800,
    n_features=8,
    covariance="block",
    noise="gene_cell",
    noise_scale=0.85,
    random_state=2026,
)
sim.X.shape, (sim.sd.min(), sim.sd.max())

The standard deviations below are known simulation inputs. They vary across both cells and features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].hist(sim.sd.ravel(), bins=50, color="#35618f")
axes[0].set(xlabel="Known measurement SD", ylabel="Cell-feature pairs")
axes[1].imshow(sim.sd[:120].T, aspect="auto", cmap="magma")
axes[1].set(xlabel="First 120 cells", ylabel="Feature", title="Heterogeneous error bars")
fig.tight_layout()

In [ ]:
naive = nc.naive_covariance(sim.X)
corrected = nc.corrected_covariance(sim.X, sim.sd, psd="clip")
mle_model = nc.HeteroscedasticGaussianCovariance(max_iter=80, tolerance=1e-6).fit(sim.X, sim.sd)
factor_model = nc.HeteroscedasticFactorAnalysis(2, max_iter=60, tolerance=2e-5).fit(sim.X, sim.sd)
estimates = {
    "true": sim.Sigma_true,
    "naive": naive,
    "corrected": corrected,
    "MLE": mle_model.covariance_,
    "factor": factor_model.covariance_,
}
pd.DataFrame(
    {
        name: {
            "relative Frobenius error": nc.relative_frobenius_error(value, sim.Sigma_true),
            "correlation RMSE": nc.correlation_rmse(value, sim.Sigma_true),
        }
        for name, value in estimates.items()
        if name != "true"
    }
).T

In [ ]:
fig, axes = plt.subplots(2, len(estimates), figsize=(14, 5.2))
limit = max(np.abs(value).max() for value in estimates.values())
for column, (name, value) in enumerate(estimates.items()):
    axes[0, column].imshow(value, cmap="coolwarm", vmin=-limit, vmax=limit)
    axes[0, column].set_title(name)
    axes[1, column].imshow(nc.covariance_to_correlation(value), cmap="coolwarm", vmin=-1, vmax=1)
    for axis in axes[:, column]:
        axis.set_xticks([])
        axis.set_yticks([])
axes[0, 0].set_ylabel("Covariance")
axes[1, 0].set_ylabel("Correlation")
fig.tight_layout()

Naive off-diagonal covariance is unbiased under independent errors, but naive correlation is attenuated because its diagonal denominators are inflated.

In [ ]:
noise_rows = []
for noise_scale in [0.0, 0.3, 0.6, 0.9, 1.2]:
    sample = nc.simulate_measurement_error(2000, 8, noise_scale=noise_scale, random_state=11)
    noise_rows.append(
        {
            "noise scale": noise_scale,
            "naive": nc.relative_frobenius_error(nc.naive_covariance(sample.X), sample.Sigma_true),
            "corrected": nc.relative_frobenius_error(
                nc.corrected_covariance(sample.X, sample.sd), sample.Sigma_true
            ),
        }
    )
noise_frame = pd.DataFrame(noise_rows).set_index("noise scale")
noise_frame.plot(
    marker="o", ylabel="Relative Frobenius error", title="Increasing measurement error"
)
noise_frame

In [ ]:
cell_rows = []
for n_cells in [100, 300, 1000, 3000]:
    errors = []
    for seed in range(4):
        sample = nc.simulate_measurement_error(n_cells, 8, noise_scale=0.8, random_state=seed)
        errors.append(
            nc.relative_frobenius_error(
                nc.corrected_covariance(sample.X, sample.sd), sample.Sigma_true
            )
        )
    cell_rows.append({"cells": n_cells, "mean corrected error": np.mean(errors)})
cell_frame = pd.DataFrame(cell_rows).set_index("cells")
cell_frame.plot(
    marker="o", logx=True, ylabel="Relative Frobenius error", title="Recovery improves with cells"
)
cell_frame

## Interpretation

The corrected estimator is the transparent first check. Dense MLE uses every cell-specific variance but is only feasible for moderate feature counts. The factor model regularizes the problem and avoids constructing a dense covariance until requested. Improvement here is meaningful because every estimate is scored against the same known latent covariance.